<a href="https://colab.research.google.com/github/Young-Kim-7/Deep-Learning/blob/main/RepVit_Team1_RepViT_M1_reproduction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# 1: Mount
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
# 2: GPU 확인
!nvidia-smi

Sat May 30 11:15:05 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             10W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
# 3: RepViT 코드 클론 & 패키지 설치
!git clone https://github.com/THU-MIG/RepVit.git
%cd RepVit
!pip install -r requirements.txt

Cloning into 'RepVit'...
remote: Enumerating objects: 374, done.
remote: Counting objects: 100% (102/102), done.
remote: Compressing objects: 100% (38/38), done.
remote: Total 374 (delta 81), reused 64 (delta 64), pack-reused 272 (from 1)
Receiving objects: 100% (374/374), 20.74 MiB | 36.55 MiB/s, done.
Resolving deltas: 100% (156/156), done.
/content/RepVit
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.2/50.2 kB 6.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 4.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 431.5/431.5 kB 17.9 MB/s eta 0:00:00
  Created wheel for fvcore: filename=fvcore-0.1.5.post20221221-py3-none-any.whl size=61397 sha256=b9b290c3aeb2b2e145f179d710826b0a41249bccf6be43116b6fcaa54f36e54d
  Stored in directory: /root/.cache/pip/wheels/ed/9f/a5/e4f5b27454ccd4596bd8b62432c7d6b1ca9fa22aef9d70a16a
  Created wheel for iopath: filename=iopath-0.

In [ ]:
# 4. 압축 해제
!mkdir -p /content/imagenet/val
! tar -xf /content/drive/MyDrive/ILSVRC2012_img_val.tar -C /content/imagenet/val

In [ ]:
# 5. 클래스별 폴더 정리
!wget https://raw.githubusercontent.com/soumith/imagenetloader.torch/master/valprep.sh
!mv valprep.sh /content/imagenet/val/
!cd /content/imagenet/val && bash valprep.sh

--2026-05-30 07:33:57--  https://raw.githubusercontent.com/soumith/imagenetloader.torch/master/valprep.sh
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.110.133, 185.199.108.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2220000 (2.1M) [text/plain]
Saving to: ‘valprep.sh’

valprep.sh          100%[===================>]   2.12M  --.-KB/s    in 0.05s   

2026-05-30 07:33:57 (46.4 MB/s) - ‘valprep.sh’ saved [2220000/2220000]



In [ ]:
# 6. 폴더 구조 확인
import os
val_dir = '/content/imagenet/val'
folders = [f for f in os.listdir(val_dir) if os.path.isdir(os.path.join(val_dir, f))]
print(f"클래스 폴더 수: {len(folders)}")  # 1000 이 나와야 정상

클래스 폴더 수: 1000


In [ ]:
# 7. RepVit-M1.0 Pretrained 모델 다운로드
!mkdir pretrain
!wget -P pretrain https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m1_0_distill_300e.pth

--2026-05-30 07:49:08--  https://github.com/THU-MIG/RepViT/releases/download/v1.0/repvit_m1_0_distill_300e.pth
Resolving github.com (github.com)... 140.82.116.4
Connecting to github.com (github.com)|140.82.116.4|:443... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://release-assets.githubusercontent.com/github-production-release-asset/667632599/0c651a7e-11ea-4e80-a99b-68450c7302da?sp=r&sv=2018-11-09&sr=b&spr=https&se=2026-05-30T08%3A47%3A52Z&rscd=attachment%3B+filename%3Drepvit_m1_0_distill_300e.pth&rsct=application%2Foctet-stream&skoid=96c2d410-5711-43a1-aedd-ab1947aa7ab0&sktid=398a6654-997b-47e9-b12b-9515b896b4de&skt=2026-05-30T07%3A47%3A52Z&ske=2026-05-30T08%3A47%3A52Z&sks=b&skv=2018-11-09&sig=KXLJfhZHm%2BqIwp7CdtIVxyInCarWddtsaFJUZj%2FmL5Y%3D&jwt=eyJ0eXAiOiJKV1QiLCJhbGciOiJIUzI1NiJ9.eyJpc3MiOiJnaXRodWIuY29tIiwiYXVkIjoicmVsZWFzZS1hc3NldHMuZ2l0aHVidXNlcmNvbnRlbnQuY29tIiwia2V5Ijoia2V5MSIsImV4cCI6MTc4MDEyOTE0OCwibmJmIjoxNzgwMTI3MzQ4LCJwYXRoIjoicmVsZWFzZWF

In [ ]:
# 8. 추론 실행
# dummy 이미지 파일 하나 생성
from PIL import Image
import os

os.makedirs('/content/imagenet/train/dummy', exist_ok=True)
img = Image.new('RGB', (224, 224))  # 빈 이미지 생성
img.save('/content/imagenet/train/dummy/dummy.JPEG')

# 추론 실행
!python main.py \
  --eval \
  --model repvit_m1_0 \
  --resume pretrain/repvit_m1_0_distill_300e.pth \
  --data-path /content/imagenet

Not using distributed mode
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:424: UserWarning: This DataLoader will create 10 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  self.check_worker_number_rationality()
Creating model: repvit_m1_0
number of params: 7302796
/usr/local/lib/python3.12/dist-packages/timm/utils/cuda.py:40: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  self._scaler = torch.cuda.amp.GradScaler()
Creating teacher model: regnety_160
Downloading: "https://dl.fbaipublicfiles.com/deit/regnety_160-a5fe301d.pth" to /root/.cache/torch/hub/checkpoints/regnety_160-a5fe301d.pth
100% 319M/319M [00:05<00:00, 